# Data Integration - Workforce Analytics Project
**Milestone 1: Ingestion, Cleansing, and Star-Schema Integration**

This notebook takes the cleaned dataset (output of the Data Cleaning & EDA notebook) and structures it into a queryable star-schema database for downstream modeling and dashboarding.

In [1]:
#import libraries
import pandas as pd
from sqlalchemy import create_engine

## 1. Ingest
Load the cleaned dataset (output from the Data Cleaning notebook).

In [2]:
#load cleaned dataset (already treated for outliers and inconsistent values)
raw = pd.read_csv("HR DATASET.csv")
cleaned = pd.read_csv("cleaned_attrition_dataset.csv")

print("Raw shape:", raw.shape)
print("Cleaned shape:", cleaned.shape)

Raw shape: (1470, 35)
Cleaned shape: (1470, 31)


## 2. Restore Join Key
`EmployeeNumber` was dropped during cleaning since it carries no analytical value, but it's needed here as the unique key to join fact and dimension tables. Row order is preserved by the cleaning process, so it can be safely re-attached by position.

In [3]:
df = cleaned.copy()
df.insert(0, "EmployeeNumber", raw["EmployeeNumber"].values)
df["Attrition_Flag"] = (df["Attrition"] == "Yes").astype(int)
df["OverTime_Flag"] = (df["OverTime"] == "Yes").astype(int)
df.head()

,EmployeeNumber,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,...,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition_Flag,OverTime_Flag
0,1,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,...,0.0,8.0,0.5,1,6,4.0,0.0,5.0,1,1
1,2,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,...,1.0,10.0,3.0,3,10,7.0,1.0,7.0,0,0
2,4,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,...,0.0,7.0,3.0,3,0,0.0,0.0,0.0,1,1
3,5,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,...,0.0,8.0,3.0,3,8,7.0,3.0,0.0,0,1
4,7,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,...,1.0,6.0,3.0,3,2,2.0,2.0,2.0,0,0


## 3. Model into Star Schema
Split the flat cleaned dataset into a **fact table** (measurable metrics) and **dimension tables** (descriptive attributes), so downstream teams can query cleanly instead of working off one wide flat file.

In [4]:
# dim_employee: demographic / personal attributes
dim_employee = df[[
    "EmployeeNumber", "Age", "Gender", "MaritalStatus",
    "Education", "EducationField", "DistanceFromHome"
]].drop_duplicates(subset="EmployeeNumber")

# dim_job: role / department / job-level attributes
dim_job = df[[
    "EmployeeNumber", "Department", "JobRole", "JobLevel",
    "BusinessTravel", "NumCompaniesWorked"
]].drop_duplicates(subset="EmployeeNumber")

# fact_workforce: measurable metrics, one row per employee
fact_workforce = df[[
    "EmployeeNumber", "Attrition_Flag", "MonthlyIncome", "DailyRate",
    "HourlyRate", "MonthlyRate", "PercentSalaryHike", "PerformanceRating",
    "EnvironmentSatisfaction", "JobSatisfaction", "JobInvolvement",
    "RelationshipSatisfaction", "WorkLifeBalance", "OverTime_Flag",
    "StockOptionLevel", "TotalWorkingYears", "TrainingTimesLastYear",
    "YearsAtCompany", "YearsInCurrentRole", "YearsSinceLastPromotion",
    "YearsWithCurrManager"
]]

print("dim_employee:", dim_employee.shape)
print("dim_job:", dim_job.shape)
print("fact_workforce:", fact_workforce.shape)

dim_employee: (1470, 7)
dim_job: (1470, 6)
fact_workforce: (1470, 21)


## 4. Load into SQLite Database
Write the three tables into a single database file — this becomes the shared, queryable data source for the ML/API and dashboard teams.

In [5]:
engine = create_engine("sqlite:///workforce.db")

dim_employee.to_sql("dim_employee", engine, if_exists="replace", index=False)
dim_job.to_sql("dim_job", engine, if_exists="replace", index=False)
fact_workforce.to_sql("fact_workforce", engine, if_exists="replace", index=False)

print("Loaded into workforce.db")

Loaded into workforce.db


## 5. Sanity Check
Verify the schema works correctly with a join query, and surface an initial insight: attrition rate by department.

In [6]:
query = """
SELECT j.Department, COUNT(*) AS headcount,
       SUM(f.Attrition_Flag) AS attritions,
       ROUND(100.0 * SUM(f.Attrition_Flag) / COUNT(*), 1) AS attrition_rate_pct
FROM fact_workforce f
JOIN dim_job j ON f.EmployeeNumber = j.EmployeeNumber
GROUP BY j.Department
ORDER BY attrition_rate_pct DESC;
"""

result = pd.read_sql(query, engine)
result

,Department,headcount,attritions,attrition_rate_pct
0,Sales,446,92,20.6
1,Human Resources,63,12,19.0
2,Research & Development,961,133,13.8


## 6. Deeper Insight: Attrition by Job Role
Department-level attrition hides an important pattern — breaking it down by role reveals which specific roles are actually driving attrition.

In [7]:
role_query = """
SELECT j.JobRole, COUNT(*) AS headcount,
       SUM(f.Attrition_Flag) AS attritions,
       ROUND(100.0 * SUM(f.Attrition_Flag) / COUNT(*), 1) AS attrition_rate_pct
FROM fact_workforce f
JOIN dim_job j ON f.EmployeeNumber = j.EmployeeNumber
GROUP BY j.JobRole
ORDER BY attrition_rate_pct DESC;
"""

role_result = pd.read_sql(role_query, engine)
role_result

,JobRole,headcount,attritions,attrition_rate_pct
0,Sales Representative,83,33,39.8
1,Laboratory Technician,259,62,23.9
2,Human Resources,52,12,23.1
3,Sales Executive,326,57,17.5
4,Research Scientist,292,47,16.1
5,Manufacturing Director,145,10,6.9
6,Healthcare Representative,131,9,6.9
7,Manager,102,5,4.9
8,Research Director,80,2,2.5


**Key finding:** Sales Representatives show a markedly higher attrition rate than any other role — well above even the Sales Executive role within the same department. This suggests role-level analysis is more informative for retention strategy than department-level analysis alone, and is a strong signal for the predictive/API layer built on top of this database.